# Hindi Model — Training from Scratch (0 -> 30,000 steps)

This notebook trains the Hindi GPT language model from step 0 to step 30,000.
- Dataset: manyajain111/hindi-lma-corpus
- Config: model_h_config.json (124M parameters)
- Tokenizer: hindi_tokenizer_16000.json (16,000 vocab BPE)
- Hyperparameters: Batch size 16, Gradient accumulation 4 (effective batch size 64), max steps 30,000


In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
HF_TOKEN = secrets.get_secret("HF_TOKEN")

import os
os.environ["HF_TOKEN"] = HF_TOKEN

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/CL3-410/individual-project-ManyaJain111.git"

if os.path.isdir("repo/.git"):
    print("[repo] already cloned -> pulling latest phase2 branch")
    !git -C repo checkout phase2
    !git -C repo pull origin phase2
else:
    !git clone -b phase2 {REPO_URL} repo


In [ ]:
!pip install -q huggingface_hub datasets sacrebleu rouge_score seaborn matplotlib


In [ ]:
import os, json, re, string, glob, tarfile, numpy as np
from multiprocessing import Pool, cpu_count
from huggingface_hub import hf_hub_download

EXTRACT_DIR = '/kaggle/working/lma_extracted/hindi-lma-corpus'
TOK_PATH    = 'repo/phase2/tokenizers/hindi_tokenizer_16000.json'
OUT_DIR     = '/kaggle/working'
MAX_TRAIN   = 520_000_000
CHUNK       = 5000

# 1. Download and Extract if missing
if not os.path.exists(f'{EXTRACT_DIR}/.done'):
    print("[Dataset] Downloading hindi_submission_phase1.tar.gz...")
    tar_path = hf_hub_download(repo_id="manyajain111/hindi-lma-corpus", filename="hindi_submission_phase1.tar.gz", repo_type="dataset")
    print(f"[Dataset] Extracting to {EXTRACT_DIR}...")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=EXTRACT_DIR)
    open(f'{EXTRACT_DIR}/.done', 'w').close()
    print("[Dataset] Extraction done!")
else:
    print(f"[Dataset] Already extracted in {EXTRACT_DIR}")

# 2. Tokenize function
PUNCT    = re.escape(string.punctuation + "।॥''\"\"")
WORD_PAT = re.compile(rf" ?[^{PUNCT}\s]+| ?[{PUNCT}]+", re.UNICODE)
_vocab = _merge_ranks = _b2c = _cache = None

def init_worker(tok_path):
    global _vocab, _merge_ranks, _b2c, _cache
    with open(tok_path) as f: data = json.load(f)
    _vocab = data['vocab']
    merges = [((a, b), a + b) for a, b in data['merges']]
    _merge_ranks = {p: i for i, (p, _) in enumerate(merges)}
    _b2c = {i: chr(i) for i in range(256)}
    _cache = {}

def encode_chunk(lines):
    out = []
    for line in lines:
        line = line.strip()
        if not line: continue
        for w in WORD_PAT.findall(line):
            if w not in _cache:
                s = [_b2c[b] for b in w.encode('utf-8')]
                while len(s) > 1:
                    pairs  = [(s[i], s[i+1]) for i in range(len(s)-1)]
                    ranked = [(_merge_ranks[p], i) for i, p in enumerate(pairs) if p in _merge_ranks]
                    if not ranked: break
                    _, i = min(ranked)
                    s = s[:i] + [s[i] + s[i+1]] + s[i+2:]
                _cache[w] = [_vocab.get(x, 0) for x in s]
            out.extend(_cache[w])
    return out

def tokenize_to_file(fnames, out_path, max_tok=None):
    import time
    fpaths = []
    for f in fnames:
        matches = glob.glob(f'{EXTRACT_DIR}/**/{f}', recursive=True)
        if matches: fpaths.append(matches[0])
    
    if not fpaths:
        print(f"❌ Error: Could not find any of {fnames} in {EXTRACT_DIR}")
        return

    if os.path.exists(out_path): os.remove(out_path)
    total_tokens = 0
    t0 = time.time()
    
    with open(out_path, 'wb') as f_out:
        for fpath in fpaths:
            print(f'Reading {os.path.basename(fpath)}...')
            lines = open(fpath, encoding='utf-8', errors='ignore').readlines()
            print(f'  {len(lines):,} lines | {cpu_count()} CPUs')
            chunks = [lines[i:i+CHUNK] for i in range(0, len(lines), CHUNK)]
            
            with Pool(cpu_count(), initializer=init_worker, initargs=(TOK_PATH,)) as pool:
                for i, r in enumerate(pool.imap(encode_chunk, chunks)):
                    arr = np.array(r, dtype=np.uint16)
                    f_out.write(arr.tobytes())
                    total_tokens += len(arr)
                    if i % 20 == 0:
                        tps = total_tokens / max(time.time() - t0, 1)
                        print(f'  {i}/{len(chunks)} chunks | {total_tokens:,} tokens | {tps:,.0f} tok/s', flush=True)
                    if max_tok and total_tokens >= max_tok:
                        pool.terminate() # <-- THE FIX: Kill workers immediately to prevent deadlock
                        break
            if max_tok and total_tokens >= max_tok: break
                
    print(f'✅ Saved {total_tokens:,} tokens -> {out_path}\n')

# 3. Tokenize train, val, test
tokenize_to_file(['train.txt', 'hindi_news_data.txt'], f'{OUT_DIR}/hindi_train.bin', max_tok=MAX_TRAIN)
tokenize_to_file(['val.txt'], f'{OUT_DIR}/hindi_val.bin')
tokenize_to_file(['test.txt'], f'{OUT_DIR}/hindi_test.bin')


In [ ]:
!python repo/phase2/src/train.py \
    --config repo/phase2/config/model_h_config.json \
    --tokenizer repo/phase2/tokenizers/hindi_tokenizer_16000.json \
    --dataset manyajain111/hindi-lma-corpus \
    --output_dir /kaggle/working/hindi_checkpoints \
    --batch_size 16 --grad_accum 4 \
    --max_steps 30000 --save_steps 1000 --val_steps 1000

In [ ]:
!python repo/phase2/src/evaluate.py \
    --checkpoint /kaggle/working/hindi_checkpoints/model_final.pt \
    --tokenizer repo/phase2/tokenizers/hindi_tokenizer_16000.json \
    --dataset manyajain111/hindi-lma-corpus \
    --language hindi \
    --output_dir /kaggle/working/hindi_reports

import os
from IPython.display import FileLink
!zip -r hindi_final_outputs.zip /kaggle/working/hindi_reports /kaggle/working/hindi_checkpoints/model_final.pt /kaggle/working/hindi_checkpoints/train_log.csv
print("\n✅ Zip file created successfully! Click the link below to download:")
FileLink('hindi_final_outputs.zip')